In [5]:
import torch
import torch.nn as nn
import numpy as np
from sklearn.metrics import f1_score, roc_auc_score, cohen_kappa_score
import random
import torch.nn.functional as F

from model import FF_TE, utils

seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
random.seed(seed)
train_set = FF_TE.FF_ZINC("train")
val_set = FF_TE.FF_ZINC("val")
test_set = FF_TE.FF_ZINC("test")
trn_loader = torch.utils.data.DataLoader(
    train_set,
    128,
    drop_last=True,
    shuffle=True,
    num_workers=4,
    persistent_workers=True,
)
val_loader = torch.utils.data.DataLoader(
    val_set,
    128,
    drop_last=True,
    shuffle=True,
    num_workers=4,
    persistent_workers=True,
)
tst_loader = torch.utils.data.DataLoader(
    test_set,
    128,
    drop_last=False,
    shuffle=True,
    num_workers=4,
    persistent_workers=True,
)

In [6]:
def valid(net, valid_data):
    yAll = None
    outAll = None
    outProb = None
    outembedding = None

    with torch.no_grad():
        for data in valid_data:
            x, y = data
            x = x["original_sample"].cuda()
            y = y["class_labels"].cuda()
            x = x.reshape(x.shape[0], -1)

            output = net(x)
            outembedding = utils.ts_append(outembedding, output)
            outAll = utils.ts_append(outAll, output.argmax(1))
            outProb = utils.ts_append(outProb, output)
            yAll = utils.ts_append(yAll, y)

    acc = outAll.eq(yAll).float().mean().item()
    f1_mac = f1_score(outAll.cpu().numpy(), yAll.cpu().numpy(), average="macro")
    f1_mic = cohen_kappa_score(outAll.cpu().numpy(), yAll.cpu().numpy())
    auc_s = roc_auc_score(yAll.cpu().numpy(), outProb.cpu().numpy(), multi_class="ovo")
    return (acc, f1_mac, f1_mic, auc_s, outembedding, yAll)

In [7]:
def valid_no_model(output, y):
    acc = sum(output == y) / (output.shape[0])
    f1_mac = f1_score(output, y, average="macro")
    f1_mic = cohen_kappa_score(output, y)
    auc_s = roc_auc_score(output, F.one_hot(torch.tensor(y)).numpy(), multi_class="ovo")
    return (acc, f1_mac, f1_mic, auc_s)

In [8]:
class MLPNet(nn.Module):
    def __init__(self, in_features, hidden_features, out_features):
        super(MLPNet, self).__init__()
        self.model = nn.ModuleList(
            [
                nn.Linear(in_features, hidden_features),
                nn.Linear(hidden_features, out_features),
            ]
        )
        self.relu = nn.ReLU()

    def forward(self, input):
        hidden = self.relu(self.model[0](input))
        return F.softmax(self.model[1](hidden), dim=1)

## BP-FNN

In [9]:
from tqdm import tqdm

lr = 1e-3
wd = 1e-5
epoch = 500

model = MLPNet(13, 1000, 4).cuda()
opt = torch.optim.AdamW(model.parameters(), lr=lr)

pbar = tqdm(total=epoch)
for i in range(epoch):
    pbar.set_description_str(f"Epoch: {i}/{epoch}")
    total_loss = 0
    for inputs, labels in trn_loader:
        inputs = inputs["original_sample"].cuda()
        labels = labels["class_labels"].cuda()
        opt.zero_grad()
        inputs = inputs.reshape(inputs.shape[0], -1)

        output = model(inputs)
        loss = F.nll_loss(output, labels)
        loss.backward()
        opt.step()

        total_loss += loss.item()

    trn_acc, _, _, _, _, _ = valid(model, trn_loader)
    val_acc, _, _, _, _, _ = valid(model, val_loader)
    if i % 1 == 0:
        test_acc, _, _, _, _, _ = valid(model, tst_loader)

    total_loss = total_loss / len(trn_loader)
    pbar.set_postfix(
        loss=total_loss, trn_acc=trn_acc, val_acc=val_acc, test_acc=test_acc
    )
    pbar.update(1)

with torch.no_grad():
    test_acc, test_f1_mac, test_f1_mic, test_auc, BP_FNN_output, BP_FNN_target = valid(
        model, tst_loader
    )
print(
    "test_acc:",
    test_acc,
    "f1 macro:",
    test_f1_mac,
    "CK:",
    test_f1_mic,
    "auc",
    test_auc,
)
pbar.close()

Epoch: 0/500:   0%|          | 0/500 [00:00<?, ?it/s]/ExtHDD/Users/Astroyd/SORBF/model/FF_TE.py:259: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(class_label), num_classes=self.num_classes
/ExtHDD/Users/Astroyd/SORBF/model/FF_TE.py:259: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(class_label), num_classes=self.num_classes
/ExtHDD/Users/Astroyd/SORBF/model/FF_TE.py:259: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(class_label), num_classes=self.num_classes
/ExtHDD/Users/Astroyd/SORBF/model/FF_TE.p

test_acc: 0.6940132975578308 f1 macro: 0.6776991636336248 CK: 0.5885335748616611 auc 0.8112170655484366


In [ ]:
def confusion_matrix(preds, labels, conf_matrix):
    for p, t in zip(preds, labels):
        conf_matrix[p, t] += 1
    return conf_matrix

In [ ]:
import matplotlib.pyplot as plt

bp_fnn_conf_matrix = torch.zeros(4, 4)
bp_fnn_conf_matrix = (
    confusion_matrix(
        BP_FNN_output.cpu().argmax(1), BP_FNN_target.cpu(), bp_fnn_conf_matrix
    )
    .cpu()
    .numpy()
)

labels_text = ["low", "mid-low", "mid-high", "high"]
plt.imshow(bp_fnn_conf_matrix, cmap=plt.cm.YlOrRd)

thresh = bp_fnn_conf_matrix.max() / 2
for x in range(4):
    for y in range(4):
        info = int(bp_fnn_conf_matrix[y, x])
        plt.text(
            x,
            y,
            info,
            verticalalignment="center",
            horizontalalignment="center",
            color="white" if info > thresh else "black",
            fontdict={"size": 20},
        )

plt.tight_layout()
plt.yticks(range(4), labels_text, size=18, rotation=0)
plt.xticks(range(4), labels_text, size=18, rotation=0)
plt.savefig(
    "./writing/Conf_matrices/BP_FNN.svg", bbox_inches="tight", pad_inches=0, dpi=1200
)
plt.show()
plt.close()

## FF-ARBF

In [ ]:
import torch
from config import config
from model import FF_RBF, utils

model = torch.load("./writing/saveZincLog/exp_0/output_model.pth").to(
    torch.device(config.device)
)
with torch.no_grad():
    FF_ARBF_output = []
    pos_embeddings = []
    neg_embeddings = []
    FF_ARBF_target = []
    for inputs, labels in tst_loader:
        inputs, labels = utils.preprocess_inputs(inputs, labels)

        pos_z, neg_z = model.embedding(inputs)
        pos_embeddings.append(pos_z)
        neg_embeddings.append(neg_z)
        scarlar_output = model.forward_downstream_classification_model(inputs, labels)
        FF_ARBF_output.append(scarlar_output["output"].detach())
        FF_ARBF_target.append(labels["class_labels"].detach())
    FF_ARBF_output = torch.concat(FF_ARBF_output, dim=0)
    pos_embeddings = torch.cat(pos_embeddings, dim=0)
    neg_embeddings = torch.cat(neg_embeddings, dim=0)
    FF_ARBF_target = torch.concat(FF_ARBF_target, dim=0)

    test_acc, test_f1_mac, test_f1_mic, test_auc = valid_no_model(
        FF_ARBF_output.argmax(1, keepdim=False).cpu(), FF_ARBF_target.cpu()
    )
    print(
        "test_acc:",
        test_acc,
        "f1 macro:",
        test_f1_mac,
        "CK:",
        test_f1_mic,
        "auc",
        test_auc,
    )

In [ ]:
import matplotlib.pyplot as plt

FF_ARBF_conf_matrix = torch.zeros(4, 4)
FF_ARBF_conf_matrix = (
    confusion_matrix(
        FF_ARBF_output.cpu().argmax(1), FF_ARBF_target.cpu(), FF_ARBF_conf_matrix
    )
    .cpu()
    .numpy()
)

labels_text = ["low", "mid-low", "mid-high", "high"]
plt.imshow(FF_ARBF_conf_matrix, cmap=plt.cm.YlOrRd)

thresh = FF_ARBF_conf_matrix.max() / 2
for x in range(4):
    for y in range(4):
        info = int(FF_ARBF_conf_matrix[y, x])
        plt.text(
            x,
            y,
            info,
            verticalalignment="center",
            horizontalalignment="center",
            color="white" if info > thresh else "black",
            fontdict={"size": 20},
        )

plt.tight_layout()
plt.yticks(range(4), labels_text, size=18, rotation=0)
plt.xticks(range(4), labels_text, size=18, rotation=0)
plt.savefig(
    "./writing/Conf_matrices/FF_ARBF.svg", bbox_inches="tight", pad_inches=0, dpi=1200
)
plt.show()
plt.close()

## FF-FNN

In [ ]:
import torch
from config import config
from model import FF_RBF, utils

model = torch.load("./writing/saveZincLog/exp_4/output_model.pth").to(
    torch.device(config.device)
)
with torch.no_grad():
    FF_FNN_output = []
    pos_embeddings = []
    neg_embeddings = []
    FF_FNN_target = []
    for inputs, labels in tst_loader:
        inputs, labels = utils.preprocess_inputs(inputs, labels)

        pos_z, neg_z = model.embedding(inputs)
        pos_embeddings.append(pos_z)
        neg_embeddings.append(neg_z)
        scarlar_output = model.forward_downstream_classification_model(inputs, labels)
        FF_FNN_output.append(scarlar_output["output"].detach())
        FF_FNN_target.append(labels["class_labels"].detach())
    FF_FNN_output = torch.concat(FF_FNN_output, dim=0)
    pos_embeddings = torch.cat(pos_embeddings, dim=0)
    neg_embeddings = torch.cat(neg_embeddings, dim=0)
    FF_FNN_target = torch.concat(FF_FNN_target, dim=0)

    test_acc, test_f1_mac, test_f1_mic, test_auc = valid_no_model(
        FF_FNN_output.argmax(1, keepdim=False).cpu(), FF_FNN_target.cpu()
    )
    print(
        "test_acc:",
        test_acc,
        "f1 macro:",
        test_f1_mac,
        "CK:",
        test_f1_mic,
        "auc",
        test_auc,
    )

In [ ]:
import matplotlib.pyplot as plt

FF_FNN_conf_matrix = torch.zeros(4, 4)
FF_FNN_conf_matrix = (
    confusion_matrix(
        FF_FNN_output.cpu().argmax(1), FF_FNN_target.cpu(), FF_FNN_conf_matrix
    )
    .cpu()
    .numpy()
)

labels_text = ["low", "mid-low", "mid-high", "high"]
plt.imshow(FF_FNN_conf_matrix, cmap=plt.cm.YlOrRd)

thresh = FF_FNN_conf_matrix.max() / 2
for x in range(4):
    for y in range(4):
        info = int(FF_FNN_conf_matrix[y, x])
        plt.text(
            x,
            y,
            info,
            verticalalignment="center",
            horizontalalignment="center",
            color="white" if info > thresh else "black",
            fontdict={"size": 20},
        )

plt.tight_layout()
plt.yticks(range(4), labels_text, size=18, rotation=0)
plt.xticks(range(4), labels_text, size=18, rotation=0)
plt.savefig(
    "./writing/Conf_matrices/FF_FNN.svg", bbox_inches="tight", pad_inches=0, dpi=1200
)
plt.show()
plt.close()

## VAE

In [ ]:
import torch
from config import config
from model import FF_RBF, utils

model = torch.load("./writing/saveZincLog/exp_9/output_model.pth").to(
    torch.device(config.device)
)
with torch.no_grad():
    VAE_output = []
    pos_embeddings = []
    neg_embeddings = []
    VAE_target = []
    for inputs, labels in tst_loader:
        inputs, labels = utils.preprocess_inputs(inputs, labels)

        pos_z, neg_z = model.embedding(inputs)
        pos_embeddings.append(pos_z)
        neg_embeddings.append(neg_z)
        scarlar_output = model.forward_downstream_classification_model(inputs, labels)
        VAE_output.append(scarlar_output["output"].detach())
        VAE_target.append(labels["class_labels"].detach())
    VAE_output = torch.concat(VAE_output, dim=0)
    pos_embeddings = torch.cat(pos_embeddings, dim=0)
    neg_embeddings = torch.cat(neg_embeddings, dim=0)
    VAE_target = torch.concat(VAE_target, dim=0)

    test_acc, test_f1_mac, test_f1_mic, test_auc = valid_no_model(
        VAE_output.argmax(1, keepdim=False).cpu(), VAE_target.cpu()
    )
    print(
        "test_acc:",
        test_acc,
        "f1 macro:",
        test_f1_mac,
        "CK:",
        test_f1_mic,
        "auc",
        test_auc,
    )

In [ ]:
import matplotlib.pyplot as plt

VAE_conf_matrix = torch.zeros(4, 4)
VAE_conf_matrix = (
    confusion_matrix(VAE_output.cpu().argmax(1), VAE_target.cpu(), VAE_conf_matrix)
    .cpu()
    .numpy()
)

labels_text = ["low", "mid-low", "mid-high", "high"]
plt.imshow(VAE_conf_matrix, cmap=plt.cm.YlOrRd)

thresh = VAE_conf_matrix.max() / 2
for x in range(4):
    for y in range(4):
        info = int(VAE_conf_matrix[y, x])
        plt.text(
            x,
            y,
            info,
            verticalalignment="center",
            horizontalalignment="center",
            color="white" if info > thresh else "black",
            fontdict={"size": 20},
        )

plt.tight_layout()
plt.yticks(range(4), labels_text, size=18, rotation=0)
plt.xticks(range(4), labels_text, size=18, rotation=0)
plt.savefig(
    "./writing/Conf_matrices/VAE.svg", bbox_inches="tight", pad_inches=0, dpi=1200
)
plt.show()
plt.close()

## SAE

In [ ]:
import torch
from config import config
from model import FF_RBF, utils

model = torch.load("./writing/saveZincLog/exp_5/output_model.pth").to(
    torch.device(config.device)
)
with torch.no_grad():
    SAE_output = []
    pos_embeddings = []
    neg_embeddings = []
    SAE_target = []
    for inputs, labels in tst_loader:
        inputs, labels = utils.preprocess_inputs(inputs, labels)

        pos_z, neg_z = model.embedding(inputs)
        pos_embeddings.append(pos_z)
        neg_embeddings.append(neg_z)
        scarlar_output = model.forward_downstream_classification_model(inputs, labels)
        SAE_output.append(scarlar_output["output"].detach())
        SAE_target.append(labels["class_labels"].detach())
    SAE_output = torch.concat(SAE_output, dim=0)
    pos_embeddings = torch.cat(pos_embeddings, dim=0)
    neg_embeddings = torch.cat(neg_embeddings, dim=0)
    SAE_target = torch.concat(SAE_target, dim=0)

    test_acc, test_f1_mac, test_f1_mic, test_auc = valid_no_model(
        SAE_output.argmax(1, keepdim=False).cpu(), SAE_target.cpu()
    )
    print(
        "test_acc:",
        test_acc,
        "f1 macro:",
        test_f1_mac,
        "CK:",
        test_f1_mic,
        "auc",
        test_auc,
    )

In [ ]:
import matplotlib.pyplot as plt

SAE_conf_matrix = torch.zeros(4, 4)
SAE_conf_matrix = (
    confusion_matrix(SAE_output.cpu().argmax(1), SAE_target.cpu(), SAE_conf_matrix)
    .cpu()
    .numpy()
)

labels_text = ["low", "mid-low", "mid-high", "high"]
plt.imshow(SAE_conf_matrix, cmap=plt.cm.YlOrRd)

thresh = SAE_conf_matrix.max() / 2
for x in range(4):
    for y in range(4):
        info = int(SAE_conf_matrix[y, x])
        plt.text(
            x,
            y,
            info,
            verticalalignment="center",
            horizontalalignment="center",
            color="white" if info > thresh else "black",
            fontdict={"size": 20},
        )

plt.tight_layout()
plt.yticks(range(4), labels_text, size=18, rotation=0)
plt.xticks(range(4), labels_text, size=18, rotation=0)
plt.savefig(
    "./writing/Conf_matrices/SAE.svg", bbox_inches="tight", pad_inches=0, dpi=1200
)
plt.show()
plt.close()

## MAE

In [ ]:
import torch
from config import config
from model import FF_RBF, utils

model = torch.load("./writing/saveZincLog/exp_3_best/output_model.pth").to(
    torch.device(config.device)
)
with torch.no_grad():
    MAE_output = []
    pos_embeddings = []
    neg_embeddings = []
    MAE_target = []
    for inputs, labels in tst_loader:
        inputs, labels = utils.preprocess_inputs(inputs, labels)

        pos_z, neg_z = model.embedding(inputs)
        pos_embeddings.append(pos_z)
        neg_embeddings.append(neg_z)
        scarlar_output = model.forward_downstream_classification_model(inputs, labels)
        MAE_output.append(scarlar_output["output"].detach())
        MAE_target.append(labels["class_labels"].detach())
    MAE_output = torch.concat(MAE_output, dim=0)
    pos_embeddings = torch.cat(pos_embeddings, dim=0)
    neg_embeddings = torch.cat(neg_embeddings, dim=0)
    MAE_target = torch.concat(MAE_target, dim=0)

    test_acc, test_f1_mac, test_f1_mic, test_auc = valid_no_model(
        MAE_output.argmax(1, keepdim=False).cpu(), MAE_target.cpu()
    )
    print(
        "test_acc:",
        test_acc,
        "f1 macro:",
        test_f1_mac,
        "CK:",
        test_f1_mic,
        "auc",
        test_auc,
    )

In [ ]:
import matplotlib.pyplot as plt

MAE_conf_matrix = torch.zeros(4, 4)
MAE_conf_matrix = (
    confusion_matrix(MAE_output.cpu().argmax(1), MAE_target.cpu(), MAE_conf_matrix)
    .cpu()
    .numpy()
)

labels_text = ["low", "mid-low", "mid-high", "high"]
plt.imshow(MAE_conf_matrix, cmap=plt.cm.YlOrRd)

thresh = MAE_conf_matrix.max() / 2
for x in range(4):
    for y in range(4):
        info = int(MAE_conf_matrix[y, x])
        plt.text(
            x,
            y,
            info,
            verticalalignment="center",
            horizontalalignment="center",
            color="white" if info > thresh else "black",
            fontdict={"size": 20},
        )

plt.tight_layout()
plt.yticks(range(4), labels_text, size=18, rotation=0)
plt.xticks(range(4), labels_text, size=18, rotation=0)
plt.savefig(
    "./writing/Conf_matrices/MAE.svg", bbox_inches="tight", pad_inches=0, dpi=1200
)
plt.show()
plt.close()

## GM-ORBF

In [ ]:
import torch
from config import config
from model import FF_RBF, utils

model = torch.load("./writing/saveZincLog/exp_7/output_model.pth").to(
    torch.device(config.device)
)
with torch.no_grad():
    GM_ORBF_output = []
    pos_embeddings = []
    neg_embeddings = []
    GM_ORBF_target = []
    for inputs, labels in tst_loader:
        inputs, labels = utils.preprocess_inputs(inputs, labels)

        pos_z, neg_z = model.embedding(inputs)
        pos_embeddings.append(pos_z)
        neg_embeddings.append(neg_z)
        scarlar_output = model.forward_downstream_classification_model(inputs, labels)
        GM_ORBF_output.append(scarlar_output["output"].detach())
        GM_ORBF_target.append(labels["class_labels"].detach())
    GM_ORBF_output = torch.concat(GM_ORBF_output, dim=0)
    pos_embeddings = torch.cat(pos_embeddings, dim=0)
    neg_embeddings = torch.cat(neg_embeddings, dim=0)
    GM_ORBF_target = torch.concat(GM_ORBF_target, dim=0)

    test_acc, test_f1_mac, test_f1_mic, test_auc = valid_no_model(
        GM_ORBF_output.argmax(1, keepdim=False).cpu(), GM_ORBF_target.cpu()
    )
    print(
        "test_acc:",
        test_acc,
        "f1 macro:",
        test_f1_mac,
        "CK:",
        test_f1_mic,
        "auc",
        test_auc,
    )

In [ ]:
import matplotlib.pyplot as plt

GM_ORBF_conf_matrix = torch.zeros(4, 4)
GM_ORBF_conf_matrix = (
    confusion_matrix(
        GM_ORBF_output.cpu().argmax(1), GM_ORBF_target.cpu(), GM_ORBF_conf_matrix
    )
    .cpu()
    .numpy()
)

labels_text = ["low", "mid-low", "mid-high", "high"]
plt.imshow(GM_ORBF_conf_matrix, cmap=plt.cm.YlOrRd)

thresh = GM_ORBF_conf_matrix.max() / 2
for x in range(4):
    for y in range(4):
        info = int(GM_ORBF_conf_matrix[y, x])
        plt.text(
            x,
            y,
            info,
            verticalalignment="center",
            horizontalalignment="center",
            color="white" if info > thresh else "black",
            fontdict={"size": 20},
        )

plt.tight_layout()
plt.yticks(range(4), labels_text, size=18, rotation=0)
plt.xticks(range(4), labels_text, size=18, rotation=0)
plt.savefig(
    "./writing/Conf_matrices/GM_OSRBF.svg", bbox_inches="tight", pad_inches=0, dpi=1200
)
plt.show()
plt.close()

## IOS-RBF

In [ ]:
import torch
from config import config
from model import FF_RBF, utils

model = torch.load("./writing/saveZincLog/exp_6/output_model.pth").to(
    torch.device(config.device)
)
with torch.no_grad():
    IOS_RBF_output = []
    pos_embeddings = []
    neg_embeddings = []
    IOS_RBF_target = []
    for inputs, labels in tst_loader:
        inputs, labels = utils.preprocess_inputs(inputs, labels)

        pos_z, neg_z = model.embedding(inputs)
        pos_embeddings.append(pos_z)
        neg_embeddings.append(neg_z)
        scarlar_output = model.forward_downstream_classification_model(inputs, labels)
        IOS_RBF_output.append(scarlar_output["output"].detach())
        IOS_RBF_target.append(labels["class_labels"].detach())
    IOS_RBF_output = torch.concat(IOS_RBF_output, dim=0)
    pos_embeddings = torch.cat(pos_embeddings, dim=0)
    neg_embeddings = torch.cat(neg_embeddings, dim=0)
    IOS_RBF_target = torch.concat(IOS_RBF_target, dim=0)

    test_acc, test_f1_mac, test_f1_mic, test_auc = valid_no_model(
        IOS_RBF_output.argmax(1, keepdim=False).cpu(), IOS_RBF_target.cpu()
    )
    print(
        "test_acc:",
        test_acc,
        "f1 macro:",
        test_f1_mac,
        "CK:",
        test_f1_mic,
        "auc",
        test_auc,
    )

In [ ]:
import matplotlib.pyplot as plt

IOS_RBF_conf_matrix = torch.zeros(4, 4)
IOS_RBF_conf_matrix = (
    confusion_matrix(
        IOS_RBF_output.cpu().argmax(1), IOS_RBF_target.cpu(), IOS_RBF_conf_matrix
    )
    .cpu()
    .numpy()
)

labels_text = ["low", "mid-low", "mid-high", "high"]
plt.imshow(IOS_RBF_conf_matrix, cmap=plt.cm.YlOrRd)

thresh = IOS_RBF_conf_matrix.max() / 2
for x in range(4):
    for y in range(4):
        info = int(IOS_RBF_conf_matrix[y, x])
        plt.text(
            x,
            y,
            info,
            verticalalignment="center",
            horizontalalignment="center",
            color="white" if info > thresh else "black",
            fontdict={"size": 20},
        )

plt.tight_layout()
plt.yticks(range(4), labels_text, size=18, rotation=0)
plt.xticks(range(4), labels_text, size=18, rotation=0)
plt.savefig(
    "./writing/Conf_matrices/IOS_RBF.svg", bbox_inches="tight", pad_inches=0, dpi=1200
)
plt.show()
plt.close()